**Fin 400**  
**Diether**  
**Annualizing Returns**<br>

**Overview**

+ We mostly work with monthly (or daily) data.

+ But sometimes we want quarterly or annual returns.

+ For example, plots of annual returns can be quite informative for examining portfolio performance.

+ Let's annualize some portfolios returns:

  - buyback: a portfolio that holds stocks that made large stock repurchases during the previous three years.

  -seo: a portfolio that holds stocks that performed the biggest SEOs in the previous three years.

+ Concepts and tools used: logical indexing, cumprod function, using date indexes, lagging with shift, and bar plots.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

port = pd.read_csv('http://diether.org/quant/23-port_issue.csv',parse_dates=['caldt'])
port

In [ ]:
port[['buyback','seo']].describe().round(4)

**2. Cumprod Function (Used a Few Times Now)**

+ `dataframe.cumprod()`: returns cumulative product overeach column in a dataframe.

+ If a column equals (1 + return), cumprod works like an expanding compounding function:

\begin{align*}
R_{1} &= (1+r_1) \\[1.05ex]
R_{1,2} &= (1+r_1)(1+r_2) \\[1.05ex]
R_{1,3} &= (1+r_1)(1+r_2)(1+r_3) \\[1.05ex]
\end{align*}

In [ ]:
port = port.set_index('caldt')
port.head(12)

In [ ]:
compound = (1 + port).cumprod()
compound.head(12)

**3. Calculating Annual Returns**

+ We can create annual returns by noting the we have the value of the portfolio based on investing one dollar in the portfolio at the start.

+ The annual return on a portfolio for 2016 is the following:
$$
r_{2016} = \frac{V_{Dec,2016}}{V_{Dec,215}} - 1
$$

Steps to compute annual returns:

1. Create a dataframe with only December in it. Note, if the index is a date, it has a date accessor built in. Can get the month with: df.index.month.

2. Compute the annual returns: $p0/p0.shift(1) - 1$

In [ ]:
ann = compound[compound.index.month == 12]
ann

In [ ]:
ann = ann / ann.shift(1) - 1
ann

In [ ]:
ann.describe().round(3)

**4. Bar Plots of the Annual Returns**

+ `Pandas` has a bar plot function.

+ If you don't specify the x-variable. Pandas assumes it's the index.

In [ ]:
ann['year'] = ann.index.year
ann.plot.bar(y='buyback',x='year')

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')

ann.plot.bar(y='buyback',x='year',figsize=[10,5],color='darkblue',legend=None)

In [ ]:
ann.plot.bar(y='seo',x='year',figsize=[10,5],color='teal',legend=None)

<br>

**5. Extra**

**5.1 Extra: Quarterly Returns**

+ Here is how to adjust the procedure to compute quarterly returns.

In [ ]:
mo = compound.index.month
qtr = compound[(mo == 3) | (mo == 6) | (mo == 9) | (mo == 12)]
qtr

In [ ]:
qtr = qtr / qtr.shift(1) - 1
qtr

In [ ]:
qtr.describe().round(4)

**5.2 Fewer xticks/xlabels**

Steps:

1. Save plot as plot/axes object.

2. use `set_xticks` method. This part is slightly odd because `pandas` (really matplotlib) maps the x-axis to start with 0. Have to adjust both the xticks and xlabels with range functions.

In [ ]:
ax = ann.plot.bar(y='seo',x='year',figsize=[10,5],color='teal',legend=None)
ax.set_xticks(ticks=range(0,35,2),labels=range(1990,2025,2))